# Geospatial Python
## Using Google Earth Engine to enrich data

The following notebook was initially generated using prompts in Google Colab.
This code has since been adapted to be run locally, with specific comments added for VS Code 

Instructional Slides: https://col.st/8CrKk

In [ ]:
# Connect to a Google Cloud Project
# this requires granting permissions to access your google account. 
# Best to create a new gmail account if you're uncomfortable with this at https://accounts.google.com/signin.

# With a gmail account you're comfortable with, that has a cloud project created:
# Go to https://code.earthengine.google.com/ to get your cloud project id
# See the profile icon (at top right)>Project Info
# REplace 'ee-csucentroidtest' from below as appropriate

import ee # Earth Engine Library

ee.Authenticate()

ee.Initialize(project='ee-csucentroidtest') # Replace string with your project ID

# Note, a new browser window will open allowing you to create a token.
# Once you've generated this token, copy and paste it into the top text fild.


In [ ]:
# Create an interactive map
import geemap # Google Earth Engine Map Library

map = geemap.Map()
map # show the map

In [ ]:
# Now we'll load in some data
# About the data:
#   An exploratory project looking at the variation in monarch butterfly densities across milkweed species 
#   using citizen science data. It has over 23,000 spatial-temporal data points.
#   This data comes from a combination of IMMP and MLMP surveys. The protocols are similar, and are described here:
#   https://mjv.nyc3.cdn.digitaloceanspaces.com/documents/IMMP/PROTOCOL_Activity2_2024.pdf 
#   https://mjv.nyc3.cdn.digitaloceanspaces.com/user-icons/MLMP_Activity1_RevOct2023.pdf 

import pandas as pd # We'll use the pandas library to load the data
df = pd.read_csv('data/citsci_monarchs.csv')

# Show the first 5 rows of data
print(df.head())

In [ ]:
# Show the size of the data
df.shape

In [ ]:
# prompt: create cluster plot with popup with date for locations from spreadsheet on interactive map

import folium
from folium.plugins import MarkerCluster

# Assuming 'latitude', 'longitude', 'siteid', and 'date' are columns in your DataFrame
latitudes = df['latitude']
longitudes = df['longitude']
site_ids = df['siteid']
dates = df['date']

# Create a map centered around the mean latitude and longitude
map_center = [latitudes.mean(), longitudes.mean()]
my_map = folium.Map(location=map_center, zoom_start=6)

# Create a marker cluster
marker_cluster = MarkerCluster().add_to(my_map)

# Add markers to the cluster with popups
for lat, lng, siteid, date in zip(latitudes, longitudes, site_ids, dates):
    popup_text = f"Site ID: {siteid}<br>Date: {date}"
    folium.Marker([lat, lng], popup=popup_text).add_to(marker_cluster)

# Display the map
my_map

In [ ]:
# prompt: return the date of the first row in our dataframe

# Assuming 'date' is a column containing dates in a suitable format (e.g., 'YYYY-MM-DD')
first_date = df['date'].iloc[0]
first_date

In [ ]:
# prompt: Show NDVI image from landsat 7 collection 2 on geemap at location and averaged 15 days plus or minus of date of first point in dataframe

import datetime # For working with dates

# Assuming 'date' is a column containing dates in a suitable format (e.g., 'YYYY-MM-DD')
first_date_str = df['date'].iloc[0]
first_date = datetime.datetime.strptime(first_date_str, '%Y-%m-%d').date()

# Calculate the date range (15 days before and after)
date_range_start = first_date - datetime.timedelta(days=30)
date_range_end = first_date + datetime.timedelta(days=30)

# Format dates for EE
date_range_start_str = date_range_start.strftime('%Y-%m-%d')
date_range_end_str = date_range_end.strftime('%Y-%m-%d')

# Define the region of interest (ROI) based on the first data point
first_latitude = df['latitude'].iloc[0]
first_longitude = df['longitude'].iloc[0]
roi = ee.Geometry.Point(first_longitude, first_latitude)

# Load Landsat 7 collection 2 surface reflectance data
landsat7 = ee.ImageCollection('LANDSAT/LE07/C02/T1_L2') \
    .filterBounds(roi) \
    .filterDate(date_range_start_str, date_range_end_str) \
    .select(['SR_B3', 'SR_B4'])

## Added after to destripe https://gis.stackexchange.com/questions/84319/destriping-landsat-7-images
def func_abg(image):
    filled1a = image.focal_mean(2, 'square', 'pixels', 1)
    return filled1a.blend(image)

landsat7 = landsat7.map(func_abg)


# Calculate NDVI
def addNDVI(image):
    ndvi = image.normalizedDifference(['SR_B4', 'SR_B3']).rename('NDVI')
    return image.addBands(ndvi)

landsat7_ndvi = landsat7.map(addNDVI)


# Reduce the collection by mean
mean_ndvi = landsat7_ndvi.select('NDVI').mean()

# Display the NDVI image on the map using geemap
import geemap

Map = geemap.Map()
Map.centerObject(roi, 10)
Map.addLayer(mean_ndvi, {'min': 0, 'max': 1, 'palette': ['blue', 'green', 'yellow', 'red']}, 'Mean NDVI')
Map

In [ ]:
# prompt: get NDVI from landsat 7 for each location

# test with one row
df = df[0:1]

# Function to get NDVI from Landsat 7 for a given location and date
def get_ndvi(latitude, longitude, date):
    try:
        point = ee.Geometry.Point(longitude, latitude)
        date_obj = ee.Date(date)

        # Filter Landsat 7 collection
        l7 = ee.ImageCollection('LANDSAT/LE07/C02/T1_L2') \
            .filterBounds(point) \
            .filterDate(date_obj.advance(-15, 'day'), date_obj.advance(15, 'day')) \
            .sort('CLOUD_COVER') \
            .first()
        # print(l7.getInfo(),"******")
        if l7:
            # Calculate NDVI with normalizedDifference
            # Docs https://developers.google.com/earth-engine/apidocs/ee-image-normalizeddifference
            ndvi = l7.normalizedDifference(['SR_B4', 'SR_B3']).rename('NDVI')
            ndvi_value = ndvi.reduceRegion(
                reducer=ee.Reducer.first(),
                geometry=point,
                scale=30
            ).get('NDVI').getInfo()
            return ndvi_value
        else:
            return None  # Or handle the case where no suitable image is found
    except Exception as e:
      print(f"An error occurred: {e}")
      return None


# Assuming 'latitude', 'longitude', 'siteid', and 'date' are columns in your DataFrame
# Add a new column 'NDVI' to the DataFrame
df['NDVI'] = df.apply(lambda row: get_ndvi(row['latitude'], row['longitude'], row['date']), axis=1)

# Display the updated DataFrame
print(df.head())

In [ ]:
# Save the enriched data
file_path = 'data/citsci_monarchs_enriched.csv'
df.to_csv(file_path, index=False)  # index=False prevents writing row indices to the file
